In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import scipy.io as spio
from scipy.ndimage import gaussian_filter
from scipy.ndimage.measurements import label
from scipy.spatial.distance import euclidean

def getSingleSessionPixelScaling(rawDict):
    xMax = 0
    xMin = 200
    yMax = 0
    yMin = 200
    for tr in range(len(rawDict['xPos'])):
        trxMax = np.amax(abs((rawDict['xPos'])[tr]))
        trxMin = np.amin(abs((rawDict['xPos'])[tr]))
        tryMax = np.amax(abs((rawDict['yPos'])[tr]))
        tryMin = np.amin(abs((rawDict['yPos'])[tr]))
        if trxMax > xMax:
            xMax = trxMax
        if trxMin < xMin:
            xMin = trxMin
        if tryMax > yMax:
            yMax = tryMax
        if tryMin < yMin:
            yMin = tryMin
    xScaling = 127/(xMax-xMin) #127 is from the size of the maze (0 start point)
    yScaling = 127/(yMax-yMin)
    return xScaling,yScaling,xMin,yMin

def rescalePositionValuesToCm(inputX,inputY):
    xScale,yScale,minX,minY = getSingleSessionPixelScaling({'xPos':inputX,'yPos':inputY})
    currPosX = inputX
    currPosY = inputY
    updNewPosX = [None]
    updNewPosY = [None]
    for i in range(np.shape(currPosX)[0]):
        trialPosX = np.multiply(abs(currPosX[i]),xScale)
        trialPosY = np.multiply(abs(currPosY[i]),yScale)
        updNewPosX.append(trialPosX)
        updNewPosY.append(trialPosY)
    return updNewPosX[1:],updNewPosY[1:]

def refitPositionToBoundaries(inX,inY):
    xScale,yScale,minX,minY = getSingleSessionPixelScaling({'xPos':inX,'yPos':inY})
    currPosX = inX
    currPosY = inY
    updNewPosX = [None]
    updNewPosY = [None]
    for i in range(len(currPosX)):
        trialPosX = np.multiply(abs(currPosX[i])-minX,xScale)
        trialPosY = np.multiply(abs(currPosY[i])-minY,yScale)
        updNewPosX.append(trialPosX.astype(int))
        updNewPosY.append(trialPosY.astype(int))
    return updNewPosX[1:],updNewPosY[1:]

def walkStartPosIndices(inputStartPosInd):
    outputInds = []
    for ind in inputStartPosInd:
        outputInds.append(int(ind[0][0]))
    return np.asarray(outputInds).flatten()

def getCueTrials(inputBehData):
    outCueInds = []
    for tr,arr in enumerate(inputBehData):
        if 'Cue' in arr[1]:
            outCueInds.append(tr)
    return outCueInds

def generateLimitFiles(xInput,yInput,timeInput,spInd):
    outX = [None]
    outY = [None]
    outTime = [None]
    for a,el in enumerate(spInd):
        outX.append(xInput[a][0][el:])
        outY.append(yInput[a][0][el:])
        outTime.append(timeInput[a][0][el:])
    return outX[1:],outY[1:],outTime[1:]

def generateUnitRateMaps(refX,refY,refTime,refSpikes,trialsCue,unitCount):
    cueCount = np.zeros((unitCount,128,128))
    cueTime = np.zeros((unitCount,128,128))
    spaCount = np.zeros((unitCount,128,128))
    spaTime = np.zeros((unitCount,128,128))
    for tr in range(len(refSpikes)):
        currSpikes = np.nan_to_num(refSpikes[tr],nan=0,posinf=0,neginf=0)
        currTime = refTime[tr]
        currY = refY[tr]
        currX = refX[tr]
        for ts in range(1,len(currTime)):
            lowTime = currTime[int(ts-1)]
            higTime = currTime[ts]
            deltaTime = np.asarray([(higTime-lowTime)/100000]*unitCount)
            boolSpikeTimes = np.where(np.logical_and(currSpikes>=lowTime,currSpikes<=higTime),1,0)
            sumSpikeCount = np.sum(boolSpikeTimes,axis=1)
            if tr in trialsCue:
                cueCount[:,currX[ts],currY[ts]] += sumSpikeCount
                cueTime[:,currX[ts],currY[ts]] += deltaTime
            else:
                spaCount[:,currX[ts],currY[ts]] += sumSpikeCount
                spaTime[:,currX[ts],currY[ts]] += deltaTime
    emptyCueUnits = 0
    emptySpaUnits = 0
    for un in range(np.shape(cueCount)[0]):
        currCueRate = cueCount[un]/np.where(cueTime[un]==0,1,cueTime[un])
        currCueRateRes = np.reshape(currCueRate,(1,128,128))
        currSpaRate = spaCount[un]/np.where(spaTime[un]==0,1,spaTime[un])
        currSpaRateRes = np.reshape(currSpaRate,(1,128,128))
        if un == 0:
            emptyCueUnits = currCueRateRes
            emptySpaUnits = currSpaRateRes
        else:
            emptyCueUnits = np.concatenate((emptyCueUnits,currCueRateRes),axis=0)
            emptySpaUnits = np.concatenate((emptySpaUnits,currSpaRateRes),axis=0)
    return emptyCueUnits,emptySpaUnits

def limitGaussianMap(inputArr):
    updArr = inputArr
    if np.sum(updArr) != 0:
        labeledArr,featureCt = label(updArr)
        xNonzeroInds = [None]
        yNonzeroInds = [None]
        for i in range(1,featureCt+1):
            maskArr = np.where(labeledArr==i,labeledArr,0)
            xNon,yNon = np.nonzero(maskArr)
            xNonzeroInds.append(list(xNon))
            yNonzeroInds.append(list(yNon))
        xNonzeroInds = xNonzeroInds[1:]
        yNonzeroInds = yNonzeroInds[1:]
        sumList = []
        for a in range(len(xNonzeroInds)):
            sumList.append(np.sum(updArr[xNonzeroInds[a],yNonzeroInds[a]]))
        cutoffSum = 0.5 * np.amax(sumList)
        sumList = np.where(sumList >= cutoffSum,1,0)
        for z in range(len(sumList)):
            if sumList[z] == 0:
                for x in range(len(xNonzeroInds[z])):
                    currXInd = (xNonzeroInds[z])[x]
                    currYInd = (yNonzeroInds[z])[x]
                    updArr[currXInd,currYInd] = 0
    return updArr

def smoothAndLimitFields(unitRateMaps):
    unitFields = np.zeros((1,128,128))
    for un in range(np.shape(unitRateMaps)[0]):
        currUntMap = unitRateMaps[un,:,:]
        smoothMap = gaussian_filter(currUntMap,sigma=1.2,order=0,truncate=3)
        smoothMapGaussianLimits = limitGaussianMap(smoothMap)
        smoothMapReshape = np.reshape(smoothMapGaussianLimits,(1,128,128))
        unitFields = np.concatenate((unitFields,smoothMapReshape),axis=0)
    return unitFields[1:]

def loadValuesFromPosFile(inPosFile):
    pmat = spio.loadmat(inPosFile)
    regionIndex = pmat.get('regionIndex').flatten()
    cueTrialInds = getCueTrials(pmat.get('behData'))
    startPosInd = walkStartPosIndices(pmat.get('startPosInd')[0])
    xScaled,yScaled = rescalePositionValuesToCm(pmat.get('xPos')[0],pmat.get('yPos')[0])
    xPos,yPos = refitPositionToBoundaries(xScaled,yScaled)
    vtLimX,vtLimY,vtLimTime = generateLimitFiles(xPos,yPos,pmat.get('trialTime')[0],startPosInd)
    if len(np.shape(pmat.get('spikeTimes'))) == 3:
        cue,spa = generateUnitRateMaps(vtLimX,vtLimY,vtLimTime,pmat.get('spikeTimes'),cueTrialInds,len(regionIndex))
    else:
        cue,spa = generateUnitRateMaps(vtLimX,vtLimY,vtLimTime,pmat.get('spikeTimes')[0],cueTrialInds,len(regionIndex))
    smoothedCueFields = smoothAndLimitFields(cue)
    smoothedSpaFields = smoothAndLimitFields(spa)
    return regionIndex,smoothedCueFields,smoothedSpaFields

def loadValuesFromModelFile(inModelFile):
    mmat = spio.loadmat(inModelFile)
    outDict = {'connectionAll':mmat.get('connectionAll'),'connectionHPC':mmat.get('connectionHPC'),
               'connectionPFC':mmat.get('connectionPFC'),'modelValuesAll':mmat.get('modelValuesAll').flatten(),
               'modelValuesHPC':mmat.get('modelValuesHPC').flatten(),
               'modelValuesPFC':mmat.get('modelValuesPFC').flatten()
              }
    return outDict

def handleFromPath(rawPos,rawCue,rawSpa,savCue,savSpa):
    for aRoot,aDirs,aFiles in os.walk(rawPos):
        for aFile in aFiles:
            if aFile.endswith('.mat'):
                currRawPos = os.path.join(rawPos,aFile)
                regInd,fieldsSmoothCue,fieldsSmoothSpa = loadValuesFromPosFile(currRawPos)
                currModCue = os.path.join(rawCue,aFile)
                currModSpa = os.path.join(rawSpa,aFile)
                currSavCue = os.path.join(savCue,aFile)
                currSavSpa = os.path.join(savSpa,aFile)
                try:
                    cueModelDict = loadValuesFromModelFile(currModCue)
                    cueModelDict.update({'regionIndex':regInd,'unitFields':fieldsSmoothCue})
                    savCueVar = spio.savemat(currSavCue,cueModelDict)
                except:
                    yes = 0
                try:
                    spaModelDict = loadValuesFromModelFile(currModSpa)
                    spaModelDict.update({'regionIndex':regInd,'unitFields':fieldsSmoothSpa})
                    savSpaVar = spio.savemat(currSavSpa,spaModelDict)
                except:
                    yes = 0
                print(aFile)
    return 'Done'    

warnings.filterwarnings('ignore')
rawPosP = ''
rawCueP = ''
rawSpaP = ''
savCueP = ''
savSpaP = ''
doneVar = handleFromPath(rawPosP,rawCueP,rawSpaP,savCueP,savSpaP)
print(doneVar)
